# Strava API Calling and Data Fetching

This notebook connects to the **Strava API** using the **OAuth 2.0** authorization protocol to
securely retrieve a complete history of athletic activities. The data is loaded into a **Pandas
DataFrame** for downstream analysis and can be exported to CSV or Excel.

**High-level workflow:**
1. **Import** all required libraries and configure project-wide paths and constants
2. **Authenticate** with Strava — build the authorization URL, exchange the one-time code, and manage token lifecycle
3. **Fetch** all activities from the Strava API using automatic pagination and rate-limit handling
4. **Analyze** the data inside a Pandas DataFrame

### Section 1 — Libraries & Packages

All imports are declared here so every dependency is visible at a glance before any code runs.

| Group | Packages | Purpose |
|---|---|---|
| Standard library | `os`, `pathlib`, `json`, `logging`, `time`, `urllib.parse` | File I/O, env variables, JSON serialization, logging, rate-limit delays, URL encoding |
| External | `python-dotenv` | Load API credentials from `.env` without hard-coding secrets in the notebook |
| External | `requests` | Make HTTP calls to the Strava REST API |
| External | `pandas` | Store, manipulate, and export activity data as a structured DataFrame |

In [1]:
# Libraries & Packages Import

# Built-in Python Libraries/Packages
import os
from pathlib import Path
import json
import logging
import time
from urllib.parse import urlencode

# External Libraries/Packages
from dotenv import load_dotenv
import requests
import pandas as pd

### Section 2 — Project Configuration

Defines every path and URL constant used throughout the notebook.
Centralizing configuration here means a single change propagates everywhere — no hunting for hard-coded strings.

| Constant | Description |
|---|---|
| `BASE_DIR` | Root of the project — all other paths are derived from this |
| `ENV_FILE` | Path to `.env` which stores Strava API credentials (never commit this file) |
| `TOKENS_FILE` | JSON file where OAuth tokens are persisted between notebook sessions |
| `STATE_FILE` | JSON file that tracks the last successful fetch timestamp for incremental runs |
| `LOG_DIR` / `OUTPUT_DIR` | Directories for log files and exported activity datasets |
| `CSV_PATH` / `EXCEL_PATH` / `JSON_PATH` | Full file paths for the three supported export formats |
| `STRAVA_*_URL` | Official Strava OAuth and REST API endpoint URLs |

In [2]:
# Anchor for all relative paths — keeps the notebook portable across machines.
BASE_DIR = Path.cwd()

# ── Credential & state files (project root) ──────────────────────────────────
ENV_FILE    = BASE_DIR / ".env"         # Strava API credentials (never commit this file)
TOKENS_FILE = BASE_DIR / "tokens.json"  # Persisted OAuth tokens (access + refresh)
STATE_FILE  = BASE_DIR / "state.json"   # Tracks last successful fetch for incremental runs

# ── Output directories ────────────────────────────────────────────────────────
LOG_DIR    = BASE_DIR / "logs"    # Rotating log files written during fetch runs
OUTPUT_DIR = BASE_DIR / "output"  # Exported activity datasets

# ── Export file paths ─────────────────────────────────────────────────────────
CSV_PATH   = OUTPUT_DIR / "strava_activities.csv"
EXCEL_PATH = OUTPUT_DIR / "strava_activities.xlsx"
JSON_PATH  = OUTPUT_DIR / "strava_activities.json"

# ── Strava REST API endpoints ─────────────────────────────────────────────────
STRAVA_AUTHORIZE_URL = "https://www.strava.com/oauth/authorize"
STRAVA_TOKEN_URL     = "https://www.strava.com/oauth/token"
STRAVA_API_BASE_URL  = "https://www.strava.com/api/v3"

In [4]:
# Load secrets from .env into the process environment so required_env() can read them.
load_dotenv(ENV_FILE)

# Module-level logger — attach a StreamHandler or FileHandler to see output.
log = logging.getLogger(__name__)

### Section 3 — Authentication Helper Functions

Implements the **Strava OAuth 2.0** authorization flow in three self-contained functions:

1. `required_env(name)` — safely reads a named secret from the `.env` file, failing loudly if it is missing
2. `build_auth_url()` — assembles the URL the user opens in a browser to grant the app access
3. `exchange_code(code)` — trades the short-lived authorization code for long-lived OAuth tokens and persists them to disk

#### `required_env(name)` — Safe Environment Variable Reader

Reads a named variable from the environment (populated from `.env` by `load_dotenv`).
If the variable is absent or empty, execution stops immediately with a descriptive `RuntimeError`
rather than silently passing `None` into downstream API calls.

In [5]:
def required_env(name):
    """
    Retrieve a required environment variable, raising a descriptive error if it is missing.

    A safe wrapper around ``os.getenv`` used to read Strava API secrets loaded from ``.env``
    by ``load_dotenv``. If the variable is absent or empty, execution stops immediately with
    a clear message instead of silently propagating ``None`` into API calls.

    Parameters
    ----------
    name : str
        Name of the environment variable to retrieve (e.g. ``"STRAVA_CLIENT_ID"``).

    Returns
    -------
    str
        The non-empty string value of the environment variable.

    Raises
    ------
    RuntimeError
        If the variable is not set or resolves to an empty string.

    Usage
    -----
    client_id = required_env("STRAVA_CLIENT_ID")
    """
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value

#### `build_auth_url()` — Strava Authorization URL Builder

Constructs the URL that must be opened in a browser to begin the OAuth 2.0 flow.
Strava displays a consent screen where the user logs in and grants the requested permissions,
then redirects to the configured `STRAVA_REDIRECT_URI` with a one-time `?code=` parameter
appended. That code is consumed in the next step by `exchange_code()`.

In [7]:
def build_auth_url():
    """
    Construct the Strava OAuth 2.0 authorization URL.

    Assembles a URL pointing to Strava's consent screen with the necessary query parameters.
    The user opens this URL in a browser, logs in, and clicks "Authorize". Strava then
    redirects the browser to the configured ``STRAVA_REDIRECT_URI`` with a
    ``?code=<authorization_code>`` parameter appended. That one-time code is subsequently
    passed to ``exchange_code()``.

    ``approval_prompt="force"`` is set so the consent screen appears on every call, which
    guarantees a fresh ``refresh_token`` is issued even when the app was previously authorized.

    Parameters
    ----------
    None

    Returns
    -------
    str
        Fully-formed authorization URL ready to be opened in a browser.

    Usage
    -----
    url = build_auth_url()
    print(url)  # Open in browser → authorize → copy the `code` from the redirect URL
    """
    params = {
        "client_id": required_env("STRAVA_CLIENT_ID"),
        "redirect_uri": required_env("STRAVA_REDIRECT_URI"),
        "response_type": "code",
        "approval_prompt": "force",  # Forces consent screen every time to ensure a fresh refresh token.
        "scope": os.getenv("STRAVA_SCOPE")
    }
    return f"{STRAVA_AUTHORIZE_URL}?{urlencode(params)}"

#### `exchange_code(code)` — Authorization Code → OAuth Tokens

After the user authorizes the app, Strava appends a short-lived `code` to the redirect URL.
This function sends that code to Strava's token endpoint and receives back three credentials:
- **`access_token`** — Bearer token for API requests (expires in ~6 hours)
- **`refresh_token`** — Long-lived token used to obtain a new access token after expiry
- **`expires_at`** — Unix timestamp indicating when the access token becomes invalid

The full response is written to `tokens.json` so it survives notebook restarts without requiring re-authorization.

In [8]:
def exchange_code(code):
    """
    Exchange a one-time Strava authorization code for OAuth tokens.

    After the user authorizes the app at the URL produced by ``build_auth_url()``, Strava
    appends a short-lived ``code`` to the redirect URL. This function sends that code to
    Strava's token endpoint via HTTP POST and receives back:

    - ``access_token``  — Bearer token for Strava API requests (expires in ~6 hours)
    - ``refresh_token`` — Long-lived token used to obtain new access tokens after expiry
    - ``expires_at``    — Unix timestamp indicating when the access token becomes invalid

    The full response (including athlete metadata) is written to ``tokens.json`` for reuse
    across notebook sessions.

    Parameters
    ----------
    code : str
        The one-time authorization code from Strava's redirect URL query string,
        e.g. ``"28dfaca7fc9ebcd4991e39f25db4688737631dc7"``.

    Returns
    -------
    dict
        Full token response from Strava. Key fields:
        ``access_token``, ``refresh_token``, ``expires_at``, ``token_type``, ``athlete``.

    Raises
    ------
    RuntimeError
        If Strava returns a non-200 response (invalid/expired code or wrong credentials).

    Usage
    -----
    tokens = exchange_code("28dfaca7fc9ebcd4991e39f25db4688737631dc7")
    print(tokens["access_token"])
    """
    response = requests.post(STRAVA_TOKEN_URL, data={
        "client_id": required_env("STRAVA_CLIENT_ID"),
        "client_secret": required_env("STRAVA_CLIENT_SECRET"),
        "code": code,
        "grant_type": "authorization_code",
    })

    if response.status_code != 200:
        raise RuntimeError(f"Token exchange failed: {response.text}")

    tokens = response.json()
    with open(TOKENS_FILE, "w") as f:
        json.dump(tokens, f, indent=2)

    return tokens

### Section 4 — Token Orchestration

`get_tokens` is the single entry-point that ties the two OAuth steps together:

- **Step 1** (`"auth-url"`) — prints the authorization URL to open in a browser
- **Step 2** (`"exchange"`) — exchanges the authorization code and persists the resulting tokens to `tokens.json`

After running both steps once, the tokens are reused automatically on every subsequent run.
Re-authorization is only needed if the refresh token is revoked or the requested scope changes.

In [9]:
def get_tokens(command, code=None):
    """
    Orchestrate the two-step Strava OAuth 2.0 authorization flow.

    This is the single entry-point for setting up authentication. It covers both steps:

    **Step 1 — Generate the authorization URL** (``command="auth-url"``)
        Prints the URL to open in a browser. After the user logs in and grants access,
        Strava redirects to the configured URI with ``?code=<authorization_code>`` appended.
        Copy that code and use it in Step 2.

    **Step 2 — Exchange the code for tokens** (``command="exchange"``)
        Takes the one-time authorization code, calls ``exchange_code()``, and saves
        the resulting access + refresh tokens to ``tokens.json``. Subsequent API calls
        read from this file automatically; re-authorization is only needed if the
        refresh token is revoked or the requested scope changes.

    Parameters
    ----------
    command : str
        The action to perform. Accepted values:
        - ``"auth-url"``  — Print the Strava authorization URL to stdout.
        - ``"exchange"``  — Exchange the authorization code for OAuth tokens.
    code : str, optional
        The one-time authorization code from Strava's redirect URL.
        Required when ``command="exchange"``, ignored otherwise.

    Returns
    -------
    None
        Output is printed to stdout; tokens are persisted to ``tokens.json``.

    Usage
    -----
    # Step 1: print the URL, open it in a browser, and authorize the app
    get_tokens("auth-url")

    # Step 2: paste the code from the redirect URL to obtain and save tokens
    get_tokens("exchange", code="<authorization_code_from_redirect>")
    """

    if command == "auth-url":
        print(build_auth_url())
        return

    if command == "exchange":
        tokens = exchange_code(code)
        expires_at = tokens.get("expires_at")
        print(f"Saved Strava tokens to {TOKENS_FILE}")
        if expires_at:
            print(f"Access token expires_at: {expires_at}")

In [10]:
# Step 1: Generate the Strava authorization URL.
# Open the printed URL in a browser, authorize the app, then copy the `code` value
# from the redirect URL's query string for use in the exchange step below.
get_tokens("auth-url")

https://www.strava.com/oauth/authorize?client_id=227797&redirect_uri=http%3A%2F%2Flocalhost&response_type=code&approval_prompt=force&scope=read%2Cread_all%2C+profile%3Aread_all%2C+activity%3Aread_all


In [11]:
# Step 2: Exchange the one-time authorization code for OAuth tokens.
# Replace the `code` value below with the `code` parameter from the redirect URL
# obtained after completing the authorization step above.
load_dotenv(ENV_FILE, override=True)  # Ensure .env is loaded so required_env() can read the code
auth_code = required_env("STRAVA_AUTHORIZATION_CODE")
get_tokens("exchange", code=auth_code)

Saved Strava tokens to c:\Users\wmouh\OneDrive\Documents\GitHub\Strava_Performance_Analysis\tokens.json
Access token expires_at: 1780836768


### Section 5 — Token Persistence & Auto-Refresh

Three helper functions that manage the token lifecycle after the initial authorization:

1. `load_tokens()` — reads the stored credentials from `tokens.json`
2. `save_tokens(tokens)` — writes updated credentials back to `tokens.json`
3. `get_valid_token()` — returns a ready-to-use access token, refreshing it transparently if it has expired

In [12]:
def load_tokens():
    """
    Load OAuth tokens from ``tokens.json``.

    Reads the persisted token file written by ``exchange_code()`` or updated by
    ``get_valid_token()`` after a refresh. The file must exist before this function is
    called — run ``get_tokens("auth-url")`` and ``get_tokens("exchange", code=...)``
    to create it on first use.

    Parameters
    ----------
    None

    Returns
    -------
    dict
        Token dictionary with at minimum the keys:
        ``access_token``, ``refresh_token``, ``expires_at``.

    Raises
    ------
    FileNotFoundError
        If ``tokens.json`` does not exist, with instructions on how to create it.

    Usage
    -----
    tokens = load_tokens()
    print(tokens["expires_at"])
    """
    if not TOKENS_FILE.exists():
        raise FileNotFoundError(
            f"{TOKENS_FILE} does not exist. Run `get_tokens('auth-url')`, "
            "authorize Strava, then run `get_tokens('exchange', code=<code>)`."
        )
    with open(TOKENS_FILE, "r") as f:
        return json.load(f)

In [13]:
def save_tokens(tokens):
    """
    Persist updated OAuth tokens to ``tokens.json``.

    Called by ``get_valid_token()`` after successfully refreshing an expired access token
    to ensure the latest credentials are always on disk and available for subsequent runs.

    Parameters
    ----------
    tokens : dict
        Token dictionary containing at minimum ``access_token``, ``refresh_token``,
        and ``expires_at``. Additional fields (e.g. athlete metadata) are preserved as-is.

    Returns
    -------
    None
        Writes to ``TOKENS_FILE`` in place; no return value.

    Usage
    -----
    save_tokens({"access_token": "...", "refresh_token": "...", "expires_at": 1779820336})
    """
    with open(TOKENS_FILE, "w") as f:
        json.dump(tokens, f, indent=2)

In [14]:
def get_valid_token():
    """
    Return a valid Strava access token, refreshing it automatically if it has expired.

    Loads the stored tokens from ``tokens.json`` and checks whether the access token is
    still valid. A 120-second buffer is applied so the token is proactively refreshed
    slightly before actual expiry, preventing race conditions on slow connections.

    If the token is expired, a POST request is made to Strava's token endpoint using the
    stored ``refresh_token``. The new credentials are saved back to ``tokens.json`` via
    ``save_tokens()``, keeping the file always up-to-date for future runs.

    Parameters
    ----------
    None

    Returns
    -------
    str
        A valid Bearer access token ready to include in API request headers.

    Raises
    ------
    FileNotFoundError
        If ``tokens.json`` does not exist (propagated from ``load_tokens()``).
    Exception
        If Strava's token refresh endpoint returns a non-200 response.

    Usage
    -----
    token = get_valid_token()
    headers = {"Authorization": f"Bearer {token}"}
    """
    tokens = load_tokens()

    if time.time() >= tokens["expires_at"] - 120:
        log.info("[auth] Access token expired. Refreshing...")

        response = requests.post(STRAVA_TOKEN_URL, data={
            "client_id":     required_env("STRAVA_CLIENT_ID"),
            "client_secret": required_env("STRAVA_CLIENT_SECRET"),
            "refresh_token": tokens["refresh_token"],
            "grant_type":    "refresh_token"
        })

        if response.status_code != 200:
            raise Exception(f"[auth] Token refresh failed: {response.text}")

        new_tokens = response.json()

        tokens["access_token"]  = new_tokens["access_token"]
        tokens["refresh_token"] = new_tokens["refresh_token"]
        tokens["expires_at"]    = new_tokens["expires_at"]

        save_tokens(tokens)
        log.info("[auth] Token refreshed and saved.")
    else:
        log.info("[auth] Token is valid. No refresh needed.")

    return tokens["access_token"]

### Section 6 — Activity Fetching

`fetch_all_activities` retrieves the full activity history from the Strava API using automatic
pagination. It handles two modes:

- **Full fetch** (default) — downloads the athlete's entire history from the beginning
- **Incremental fetch** — downloads only activities recorded after a given Unix timestamp, useful for updating an existing dataset efficiently

Rate-limit handling is built in: if Strava returns HTTP 429, the function pauses for 15 minutes
before retrying. A configurable `SLEEP_SEC` delay between pages keeps requests within
Strava's 100-requests/15-min and 1,000-requests/day limits.

In [15]:
# Maximum activities returned per API page (Strava's hard limit is 200).
PER_PAGE  = 200

# Seconds to wait between page requests to stay within Strava's rate limits
# (100 requests / 15 min and 1,000 requests / day).
SLEEP_SEC = 1

In [16]:
def fetch_all_activities(after_epoch=None):
    """
    Fetch all of the authenticated athlete's activities from the Strava API.

    Pages through ``GET /athlete/activities`` until Strava returns an empty page,
    collecting every activity summary object. Supports two modes:

    - **Full fetch** (default) — retrieves the athlete's entire activity history.
    - **Incremental fetch** — retrieves only activities recorded after a given Unix
      timestamp, useful for updating an existing dataset without re-downloading everything.

    Rate-limit handling: if Strava returns HTTP 429, the function waits 15 minutes before
    retrying the same page. A ``SLEEP_SEC`` pause is inserted between every successful
    page request to stay within Strava's 100-requests/15-min limit.

    Parameters
    ----------
    after_epoch : int or float, optional
        Unix timestamp (seconds since epoch). When provided, only activities recorded
        after this timestamp are returned. Defaults to ``None`` (full history fetch).

    Returns
    -------
    list of dict
        A list of Strava activity summary objects (one dict per activity).
        Pass directly to ``pd.DataFrame()`` to create a structured dataset.

    Raises
    ------
    Exception
        If the Strava API returns a non-200, non-429 HTTP status on any page request.

    Usage
    -----
    # Full fetch — retrieve entire activity history
    activities = fetch_all_activities()
    df = pd.DataFrame(activities)

    # Incremental fetch — only activities after a specific date
    import datetime
    since = datetime.datetime(2024, 1, 1).timestamp()
    activities = fetch_all_activities(after_epoch=since)
    """
    token      = get_valid_token()
    headers    = {"Authorization": f"Bearer {token}"}
    activities = []
    page       = 1
    params     = {"per_page": PER_PAGE}

    if after_epoch:
        params["after"] = int(after_epoch)
        log.info("[fetch] Starting incremental activity fetch after %s.", after_epoch)
    else:
        log.info("[fetch] Starting full activity fetch...")

    while True:
        params["page"] = page
        response = requests.get(
            f"{STRAVA_API_BASE_URL}/athlete/activities",
            headers=headers,
            params=params
        )

        if response.status_code == 429:
            log.warning("[fetch] Rate limit hit. Waiting 15 minutes...")
            time.sleep(900)
            continue

        if response.status_code != 200:
            raise Exception(f"[fetch] API error: {response.text}")

        batch = response.json()

        if not batch:
            log.info("[fetch] No more activities. Done at page %s.", page - 1)
            break

        activities.extend(batch)
        log.info(
            "[fetch] Page %s - %s activities fetched (total so far: %s)",
            page,
            len(batch),
            len(activities),
        )

        page += 1
        time.sleep(SLEEP_SEC)

    log.info("[fetch] Total activities fetched: %s", len(activities))
    return activities

### Section 7 — Load Data into DataFrame

Calls `fetch_all_activities()` to pull the complete activity history and loads the result
directly into a Pandas DataFrame. Each row represents one activity; columns correspond to
the fields returned by the Strava API (e.g. `name`, `type`, `distance`, `moving_time`,
`start_date`, `average_speed`, `total_elevation_gain`, etc.).

In [17]:
# Fetch all activities and load them into a Pandas DataFrame.
# Each row is one activity; columns are the fields returned by the Strava API.
df = pd.DataFrame(fetch_all_activities())
df

,resource_state,athlete,name,distance,moving_time,elapsed_time,total_elevation_gain,type,sport_type,workout_type,...,upload_id,upload_id_str,external_id,from_accepted_tag,pr_count,total_photo_count,has_kudoed,suffer_score,average_cadence,device_name
0,2,"{'id': 843017438, 'resource_state': 1}",HIIT,0.0,245,245,0.0,Workout,HighIntensityIntervalTraining,NaN,...,19929048363,19929048363,1780808073965.fit,False,0,0,False,3.0,NaN,NaN
1,2,"{'id': 843017438, 'resource_state': 1}",Indoor run,1070.0,1035,1035,0.0,Run,Run,NaN,...,19929022436,19929022436,1780807647237.fit,False,0,0,False,13.0,55.1,NaN
2,2,"{'id': 843017438, 'resource_state': 1}",Indoor walk,750.0,1075,1075,0.0,Walk,Walk,NaN,...,19928777710,19928777710,1780803732842.fit,False,0,0,False,6.0,54.2,NaN
3,2,"{'id': 843017438, 'resource_state': 1}",Indoor run,1240.0,1185,1185,0.0,Run,Run,NaN,...,19928656976,19928656976,1780802055102.fit,False,0,0,False,34.0,56.4,NaN
4,2,"{'id': 843017438, 'resource_state': 1}",Indoor walk,950.0,930,930,0.0,Walk,Walk,NaN,...,19928656947,19928656947,1780802054256.fit,False,0,0,False,14.0,55.7,NaN
5,2,"{'id': 843017438, 'resource_state': 1}",Indoor run,700.0,905,905,0.0,Run,Run,NaN,...,19895391220,19895391220,1780586723508.fit,False,0,0,False,4.0,52.8,NaN
6,2,"{'id': 843017438, 'resource_state': 1}",Indoor walk,650.0,915,915,0.0,Walk,Walk,NaN,...,19894838458,19894838458,1780583988653.fit,False,0,0,False,2.0,54.2,NaN
7,2,"{'id': 843017438, 'resource_state': 1}",Outdoor walk,10263.4,6036,6160,79.4,Walk,Walk,NaN,...,19875070186,19875070186,1780462662492.fit,False,0,0,False,22.0,57.8,NaN
8,2,"{'id': 843017438, 'resource_state': 1}",Outdoor walk,10277.0,6087,6395,79.6,Walk,Walk,NaN,...,19860482229,19860482229,1780376196575.fit,False,0,0,False,23.0,57.5,NaN
9,2,"{'id': 843017438, 'resource_state': 1}",HIIT,0.0,1005,1005,0.0,Workout,HighIntensityIntervalTraining,NaN,...,19736833077,19736833077,1779601140871.fit,False,0,0,False,15.0,NaN,NaN


### Section 8 — Export Data to JSON

Exports the activities DataFrame to a JSON file stored in the `output/` directory.
Each activity is written as an object inside a top-level JSON array (`orient="records"`),
making the file easy to consume in any language or tool. Dates are serialized in ISO 8601
format and Unicode characters (e.g. accented names) are preserved as-is.

In [18]:
def export_to_json(df, path=JSON_PATH):
    """
    Export a Pandas DataFrame to a JSON file.

    Serializes the DataFrame as a JSON array where each element is one activity object
    (``orient="records"``). The output directory is created automatically if it does not
    exist. Dates are written in ISO 8601 format and Unicode characters are preserved
    without escaping (``force_ascii=False``).

    Parameters
    ----------
    df : pandas.DataFrame
        The activities DataFrame to export. Typically the ``df`` produced by
        ``pd.DataFrame(fetch_all_activities())``.
    path : pathlib.Path or str, optional
        Destination file path. Defaults to ``JSON_PATH``
        (``<project_root>/output/strava_activities.json``).

    Returns
    -------
    None
        Writes the file to ``path`` and prints a confirmation message.

    Usage
    -----
    # Export using the default path defined in project configuration
    export_to_json(df)

    # Export to a custom path
    export_to_json(df, path="my_folder/activities.json")
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    df.to_json(
        path,
        orient="records",
        indent=2,
        date_format="iso",
        force_ascii=False,
    )

    print(f"Exported {len(df)} activities to {path}")

In [19]:
export_to_json(df)

Exported 55 activities to c:\Users\wmouh\OneDrive\Documents\GitHub\Strava_Performance_Analysis\output\strava_activities.json
